# Classifier catalog experiments

The classifier uses the `/v1/systemone` API and currently serves `jiwo-4b`. Configure its address and model through `CLASSIFIER_BASE_URL` and `CLASSIFIER_MODEL` in the repository-root `.env`. No local GPU or model packages are needed.

Run the offline setup first. Set `RUN_LIVE = True` only when the configured service is available. Experiment calls run sequentially. All criteria and instructions follow the service contract.

The labeled cases are hand-authored diagnostics, not held-out final-code accuracy. We compare an exact-value baseline before model-assisted extraction. Acceptance criteria: correct negation/unknown/conflict handling, correct numeric ownership, and narrowing only on supported facts. Report counts and failures alongside accuracy; do not promote the default confidence threshold without calibration.

Stored outputs from earlier runs retain their original model IDs. Rerun the relevant cells to produce outputs for the current classifier.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import json
import logging
import sys
from collections import Counter

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').exists() and (p / 'experiments').is_dir())
sys.path.insert(0, str(ROOT / 'experiments'))
import classifier_experiments as ce
from config import Settings

settings = Settings.from_environment()
logging.basicConfig(level=settings.log_level.upper())
BASE_URL = settings.classifier_base_url
RUN_LIVE = False
TIMEOUT_SECONDS = settings.classifier_timeout_seconds
THRESHOLD = settings.threshold  # uncalibrated experiment parameter
client = ce.ClassifierClient(BASE_URL, TIMEOUT_SECONDS)
print('Project:', ROOT)
print('Model:', ce.MODEL)
print('Live calls enabled:', RUN_LIVE)

Project: /home/zolotarev/PycharmProjects/hackathon-tenderhack-kazan-2026-pitonomeshalka
Model: bespokelabs/Bespoke-Nimble-9B-v3
Live calls enabled: False


In [2]:
cache = settings.catalog_path
REBUILD_CATALOG = False  # set True after changing the source snapshot
if cache.exists() and not REBUILD_CATALOG:
    catalog = json.loads(cache.read_text())
else:
    catalog = ce.load_catalog(settings.xml_dir)
    cache.parent.mkdir(parents=True, exist_ok=True)
    cache.write_text(json.dumps(catalog, ensure_ascii=False) + '\n')

print('XML parts read:', catalog['xml_files'])
print('Exploratory ACTIVE non-template positions:', len(catalog['positions']))
print('Deduplicated categorical facts:', len(catalog['facts']))
for scope in ('drills', 'clothing', 'furniture', 'tools', 'all'):
    positions = ce.select_scope(catalog, scope)
    facts = ce.family_facts(catalog, positions)
    print(scope, 'positions=', len(positions), 'facts=', len(facts))
print('Drill codes:', [(p['code'], p['version']) for p in ce.drill_family(catalog)])
print('Scope uses rubrics 116/128 and tool prefixes 25.73/28.24; no full workbook crosswalk or date eligibility policy is claimed.')

XML parts read: 82
Exploratory ACTIVE non-template positions: 1302
Deduplicated categorical facts: 2568
drills positions= 2 facts= 20
clothing positions= 922 facts= 822
furniture positions= 190 facts= 936
tools positions= 190 facts= 820
all positions= 1302 facts= 2568
Drill codes: [('28.24.11.000-00000002', '4'), ('28.24.11.000-00000001', '5')]
Scope uses rubrics 116/128 and tool prefixes 25.73/28.24; no full workbook crosswalk or date eligibility policy is claimed.


In [3]:
TEXT = 'Нужна аккумуляторная дрель 18 В с реверсом, без ударного режима.'
for suite in ('smoke', 'numeric', 'order', 'slots', 'facts', 'progressive'):
    jobs = ce.build_jobs(catalog, suite, TEXT, chunk_size=4)
    print(suite, len(jobs), 'requests;', sum(len(j['body']['questions']) for j in jobs), 'questions')
print(json.dumps(ce.build_jobs(catalog, 'facts', TEXT)[0]['body'], ensure_ascii=False, indent=2))

records_by_suite = {}
summary_by_suite = {}
def run_suite(suite, text=TEXT, chunk_size=4, scope='drills'):
    if not RUN_LIVE:
        print('Offline mode: set RUN_LIVE=True after configuring the classifier connection.')
        return None
    jobs = ce.build_jobs(catalog, suite, text, chunk_size, scope)
    destination = ROOT / 'experiments/results' / (datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ') + '_' + suite)
    records, summary = ce.run_jobs(client, jobs, destination, catalog, THRESHOLD, scope)
    records_by_suite[suite] = records
    summary_by_suite[suite] = summary
    print('Saved:', destination)
    print(json.dumps(summary['experiments'], ensure_ascii=False, indent=2))
    return summary

smoke 16 requests; 16 questions
numeric 6 requests; 6 questions
order 3 requests; 3 questions
slots 9 requests; 9 questions
facts 5 requests; 20 questions
progressive 20 requests; 80 questions
{
  "model": "bespokelabs/Bespoke-Nimble-9B-v3",
  "state": "Нужна аккумуляторная дрель 18 В с реверсом, без ударного режима.",
  "questions": {
    "0dd52e285246f223dd87": {
      "type": "choice",
      "instructions": "Проверь факт о самом товаре: Наличие режима регулировки оборотов = Да. Выбери supported, contradicted, unknown или conflicting. Не переноси свойства упаковки на товар. Не додумывай отсутствующие данные.",
      "criteria": {
        "supported": "Описание явно подтверждает факт, включая однозначные синонимы.",
        "contradicted": "Описание явно отрицает факт или прямо указывает несовместимое значение.",
        "unknown": "Для проверки факта недостаточно данных. Отсутствие упоминания не означает отрицание.",
        "conflicting": "В описании есть одновременно подтверждение 

## 1. Health and labeled smoke cases

The 16 cases cover product-domain routing, explicit yes/no, missing information, inconsistent input, typos, and packaging evidence. The exact-value baseline is intentionally conservative and has no synonym or negation model. Baseline and model scores for fact-status cases share the same diagnostic labels; domain routing is reported separately in the confusion rows. The server's `/health` returns configuration details but does not prove a scoring request will succeed.

In [4]:
smoke = ce.smoke_jobs()
baseline_only = [dict(job, error='model not run') for job in smoke]
print('Exact-value baseline:', ce.summarize(baseline_only)['exact_value_baseline'])
if RUN_LIVE:
    print(json.dumps(client.health(), ensure_ascii=False, indent=2))
    smoke_summary = run_suite('smoke')
    print(json.dumps(smoke_summary['exact_value_baseline'], indent=2))
    print(json.dumps(smoke_summary['confusion'], ensure_ascii=False, indent=2))
else:
    print('No API calls made.')

Exact-value baseline: {'correct': 3, 'labeled': 12, 'accuracy': 0.25}
No API calls made.


## 2. Numeric binding and normalization

The model chooses a candidate span from the input; code normalizes its number and unit. Cases distinguish the battery's 18 V from the charger's 220 V, convert 0.018 kV to 18 V, and handle missing/contradictory values. This is a focused voltage experiment, not a complete extractor for arbitrary ranges, scientific notation or all OKEI units.

In [5]:
for job in ce.numeric_jobs():
    print(job['id'], 'expected=', job['expected'], 'spans=', [(s['raw'], s['value'], s['okei']) for s in job['spans']])
numeric_summary = run_suite('numeric')
if numeric_summary is not None:
    for record in records_by_suite['numeric']:
        print(record['id'], record.get('bound_measurement'), record.get('error'))

voltage_exact expected= {'voltage': 'span_0'} spans= [('18 В', '18', '222')]
voltage_conversion expected= {'voltage': 'span_0'} spans= [('0,018 кВ', '18.000', '222')]
charger expected= {'voltage': 'span_0'} spans= [('18 В', '18', '222'), ('220 В', '220', '222')]
charger_only expected= {'voltage': 'unknown'} spans= [('220 В', '220', '222')]
voltage_conflict expected= {'voltage': 'conflicting'} spans= [('18 В', '18', '222'), ('12 В', '12', '222')]
voltage_missing expected= {'voltage': 'unknown'} spans= []
Offline mode: set RUN_LIVE=True after opening the tunnel.


## 3. Catalog slots versus independent fact checks

The drill family is read from source XML: all its categorical values are supplied to the model. Slots select one catalog value or unknown/conflicting. Independent checks classify every candidate fact; several facts can be supported. Both approaches have the same description and vocabulary. Slot IDs decode through each request's `facts` metadata. Multi-valued properties should use independent checks.

In [6]:
slot_summary = run_suite('slots')
fact_summary = run_suite('facts')
if fact_summary is not None:
    narrowed = fact_summary['narrowing']['facts_0']
    print('Complete categorical scan:', narrowed['complete_fact_scan'])
    print(json.dumps(narrowed['candidates'], ensure_ascii=False, indent=2))
    for threshold in (0.6, 0.8, 0.95):
        candidates = ce.narrow_candidates(ce.drill_family(catalog), ce.family_facts(catalog, ce.drill_family(catalog)), records_by_suite['facts'], threshold)
        print('Uncalibrated threshold', threshold, 'compatible codes:', [c['code'] for c in candidates if c['compatible']])

Offline mode: set RUN_LIVE=True after opening the tunnel.
Offline mode: set RUN_LIVE=True after opening the tunnel.


## 4. Progressive narrowing

A fixed drill candidate pool is rescored as the input adds battery power, reverse and absence of impact mode. Missing information never eliminates a candidate. Allowed catalog values are permissions, not assertions: a candidate allowing both reverse/no reverse is not excluded merely because one hypothesis is contradicted. Returned compatibility is exploratory categorical filtering; required fields, numeric intervals, lifecycle metadata and object relevance still need validation before any final-code claim.

In [7]:
progressive_summary = run_suite('progressive')
if progressive_summary is not None:
    for step, result in progressive_summary['narrowing'].items():
        print(step, 'complete=', result['complete_fact_scan'], 'compatible=', [c['code'] for c in result['candidates'] if c['compatible']])

Offline mode: set RUN_LIVE=True after opening the tunnel.


## 5. Repeatability, option order and question chunking

The team's handler scores each question separately, with a full prompt for each field. Requesting many questions together does not imply a shared-backbone single forward pass. The order probe reverses criteria and repeats an identical request. The optional chunk probe changes only the number of questions per HTTP request (1, 4, 8); compare choices, reported input tokens and elapsed time. These measurements depend on shared-GPU load.

In [8]:
order_summary = run_suite('order')
if order_summary is not None:
    print(json.dumps(order_summary.get('order_sensitivity'), indent=2))

RUN_CHUNK_COMPARISON = False
if RUN_LIVE and RUN_CHUNK_COMPARISON:
    chunk_records = {}
    for size in (1, 4, 8):
        summary = run_suite('facts', chunk_size=size)
        chunk_records[size] = list(records_by_suite['facts'])
        tokens = sum(r.get('response', {}).get('usage', {}).get('input_tokens', 0) for r in chunk_records[size])
        seconds = sum(r.get('elapsed_seconds', 0) for r in chunk_records[size])
        print('chunk=', size, 'input_tokens=', tokens, 'seconds=', seconds)
    predictions = {size: {qid: a['choice'] for r in records for qid, a in r.get('response', {}).get('answers', {}).items()} for size, records in chunk_records.items()}
    for size in (4, 8):
        common = predictions[1].keys() & predictions[size].keys()
        print('chunk=', size, 'changed choices=', sum(predictions[1][q] != predictions[size][q] for q in common), 'compared=', len(common))

Offline mode: set RUN_LIVE=True after opening the tunnel.


## 6. Explore the larger catalog and real STE descriptions

All 2,568 deduplicated categorical facts from the selected domain subsets can be used as an exhaustive offline baseline. This is much more inference work than the 20-fact drill pilot. The full scan is opt-in and not automatically executed. Excel STE examples are unlabelled: use them for inspection, not final-code accuracy. The existing relationship audit contains source-row examples; input characteristics can include suspicious zero values and packaging measurements.

In [9]:
full_jobs = ce.build_jobs(catalog, 'facts', TEXT, chunk_size=4, scope='all')
print('Full exploratory vocabulary:', len(full_jobs), 'requests;', sum(len(j['body']['questions']) for j in full_jobs), 'questions')
RUN_FULL_CATALOG = False
if RUN_LIVE and RUN_FULL_CATALOG:
    run_suite('facts', scope='all')

# Inspect a real, unlabelled STE input saved by the earlier source audit.
audit_path = ROOT / 'docs/characteristic-first-audit.json'
if audit_path.exists():
    audit = json.loads(audit_path.read_text())
    for workbook, info in audit['workbooks'].items():
        sample = info['ste_samples'][0]
        print(workbook, 'row=', sample['row'], 'id=', sample['id'])
        print(sample['name'])
        print(sample['characteristics'][:600])
else:
    print('Optional prior audit not found; synthetic and XML catalog experiments remain runnable.')

CUSTOM_TEXT = TEXT  # replace with a real STE name + raw characteristics
CUSTOM_SCOPE = 'drills'  # furniture/clothing/tools/all are also available
RUN_CUSTOM = False
if RUN_LIVE and RUN_CUSTOM:
    run_suite('facts', text=CUSTOM_TEXT, scope=CUSTOM_SCOPE)

Full exploratory vocabulary: 642 requests; 2568 questions
Инструменты row= 2 id= 1160207
Дрель-шуруповерт аккумуляторная Ставр "ДА-10,8/2 (2,6 Л)"
Вес 2.03000 кг | Высота упаковки 100 мм | Глубина упаковки 330 мм | Диаметр сверления (металл) 6 мм | Дополнительные функции Блокировка шпинделя,Блокировка от случайного включения,Бесступенчатая регулировка частоты вращения | Комплектация Дрель-шуруповерт, зарядное устройство, 6 сверл, 6 бит, переходник для бит, аккумулятор, пластиковый кейс | Конструктивные особенности Рукоятки с резиновым покрытием,Защитный кожух,Индикатор заряда аккумулятора | Максимальные обороты на 1ой скорости 350 об/мин | Максимальные обороты на 2ой скорости 1300 об/мин | Максимальный диаметр зажима патрона 0.00000 мм |
Мебель row= 2 id= 1193874
Стойка для рекламных материалов напольная, сетчатая, на 32 лотка, вращающаяся, белая
Масса 10.20000 кг | Тип стойка
Одежда row= 2 id= 1174868
Рюкзак Walker Pure Authentic цвет синий
Бренд Walker | Вес 0.00000 кг | Высота упако

Results are saved under `experiments/results/<UTC timestamp>_<suite>/`: health configuration, manifest (including catalog hash when used), raw requests/responses and timings, and summary. Failed requests are recorded and stop the run; no automatic retries are made because a timed-out inference can still be occupying the shared server. Health failure is saved separately. Read `experiments/README.md` for command-line use and limits of these diagnostics.

## 7. Review saved runs before changing the experiment

This section reads existing artifacts without contacting the API. Accuracy is shown only when labels were saved; unlabeled slots/facts/progressive runs must not be treated as zero accuracy. See `experiments/results-analysis.md` for the manual review and its limits. The continuation targets numeric span selection, unsupported additions, contradictions and explicit corrections.

In [10]:
saved_runs = []
saved_records = []
for directory in sorted((ROOT / 'experiments/results').iterdir()):
    summary_path = directory / 'summary.json'
    log_path = directory / 'requests.jsonl'
    if not summary_path.exists() or not log_path.exists():
        continue
    summary = json.loads(summary_path.read_text())
    records = [json.loads(line) for line in log_path.read_text().splitlines() if line.strip()]
    saved_records.extend(records)
    for suite, metrics in summary['experiments'].items():
        accuracy = metrics.get('accuracy')
        saved_runs.append({
            'run': directory.name, 'suite': suite,
            'requests': metrics['requests'], 'errors': metrics['errors'],
            'labeled': metrics['labeled'],
            'accuracy': f"{accuracy:.1%}" if accuracy is not None else 'unlabeled',
            'seconds': round(sum(r.get('elapsed_seconds', 0) for r in records if r['experiment'] == suite), 2),
        })

print('Saved runs:', len(saved_runs))
print(f"{'run':46} {'requests':>8} {'errors':>6} {'labels':>6} {'accuracy':>10} {'seconds':>8}")
for row in saved_runs:
    print(f"{row['run']:46} {row['requests']:8} {row['errors']:6} {row['labeled']:6} {row['accuracy']:>10} {row['seconds']:8.2f}")

print('Saved labeled failures:')
for record in saved_records:
    for question_id, expected in record.get('expected', {}).items():
        answer = record.get('response', {}).get('answers', {}).get(question_id)
        if answer and answer['choice'] != expected:
            print(record['id'], 'expected=', expected, 'predicted=', answer['choice'], 'probabilities=', answer['probabilities'])
if not saved_runs:
    print('No saved completed runs; run the baseline suites when the API is available.')

Saved runs: 7
run                                            requests errors labels   accuracy  seconds
20261010T091105033983Z_smoke                         16      0     16      93.8%     8.96
20261010T091131549904Z_numeric                        6      0      6      83.3%     3.55
20261010T091135568053Z_slots                          9      0      0  unlabeled     5.47
20261010T091141409926Z_facts                          5      0      0  unlabeled    13.83
20261010T091155639556Z_progressive                   20      0      0  unlabeled    46.28
20261010T091242236665Z_order                          3      0      3     100.0%     1.46
20261010T091854823729Z_facts                          5      0      0  unlabeled    10.15
Saved labeled failures:
power_conflict expected= conflicting predicted= contradicted probabilities= {'supported': 0.009149826828080927, 'contradicted': 0.5074306002126306, 'unknown': 0.01777507958820698, 'conflicting': 0.4656444933710813}
voltage_conversion expected

## 8. Focused follow-up diagnostics

The numeric A/B test repeats the same six labeled inputs. It changes only the voltage candidate descriptions: the normalized arm adds the deterministic value in volts and OKEI 222 while preserving raw text, offsets, options, instructions and labels. This tests representation, not a new extraction model.

The fact checks cover tools, furniture and clothing with explicit support, missing information, negation, packaging, unresolved contradictions and explicit corrections. These are manually labeled development fixtures. Target metrics are status accuracy, supported-fact precision, supported-fact recall and unsupported additions, including accepted additions at the uncalibrated threshold. Desired behavior: no supported predictions on unknown/contradicted/conflicting labels; report failures rather than tuning a threshold on these cases.

A fresh follow-up run makes 29 sequential HTTP requests (12 numeric + 17 fact checks), stops on the first API error, and saves raw responses and a separate metric report. It is gated by both `RUN_LIVE` and `RUN_FOLLOWUP`.

In [11]:
from copy import deepcopy

numeric_ab_jobs = []
for original in ce.numeric_jobs():
    for arm in ('raw', 'normalized'):
        job = deepcopy(original)
        job.update(id=original['id'] + '_' + arm, experiment='numeric_' + arm, pair_id=original['id'], arm=arm)
        if arm == 'normalized':
            criteria = job['body']['questions']['voltage']['criteria']
            for span in job['spans']:
                if span['dimension'] == 'voltage':
                    criteria[span['id']] += f" Нормализованное значение: {span['value']} В; ОКЕИ {span['okei']}."
        numeric_ab_jobs.append(job)

followup_cases = [
    ('tool_missing_impact', 'tools', 'Нужна дрель.', 'Наличие режима ударного сверления', 'Да', 'unknown'),
    ('tool_impact_no', 'tools', 'Нужна аккумуляторная дрель с реверсом, без ударного режима.', 'Наличие режима ударного сверления', 'Нет', 'supported'),
    ('tool_reverse_no', 'tools', 'Нужна аккумуляторная дрель с реверсом.', 'Наличие режима реверса', 'Нет', 'contradicted'),
    ('tool_missing_battery_count', 'tools', 'Нужна аккумуляторная дрель.', 'Количество аккумуляторов в комплекте', '1', 'unknown'),
    ('tool_missing_angle', 'tools', 'Нужна аккумуляторная дрель.', 'Конструкция', 'Угловая', 'unknown'),
    ('tool_power_conflict', 'tools', 'В двух действующих разделах указаны взаимоисключающие требования к одной дрели: только от аккумулятора; только от сети. Оба требования остаются в силе.', 'Тип питания', 'От аккумулятора', 'conflicting'),
    ('tool_power_correction', 'tools', 'Дрель от сети. Исправление: предыдущее требование отменено, нужна дрель только от аккумулятора.', 'Тип питания', 'От аккумулятора', 'supported'),
    ('furniture_explicit', 'furniture', 'Нужен стул с подлокотниками.', 'Наличие подлокотников', 'Да', 'supported'),
    ('furniture_missing', 'furniture', 'Нужен деревянный стул.', 'Наличие подлокотников', 'Да', 'unknown'),
    ('furniture_negation', 'furniture', 'Нужен стул без подлокотников.', 'Наличие подлокотников', 'Да', 'contradicted'),
    ('furniture_package', 'furniture', 'На коробке указан стул с подлокотниками. Сам стул поставляется без подлокотников.', 'Наличие подлокотников', 'Да', 'contradicted'),
    ('furniture_conflict', 'furniture', 'Для одного стула действуют два несовместимых требования: наличие подлокотников — да; наличие подлокотников — нет. Ни одно не отменено.', 'Наличие подлокотников', 'Да', 'conflicting'),
    ('clothing_explicit', 'clothing', 'Нужна куртка из хлопка.', 'Материал', 'Хлопок', 'supported'),
    ('clothing_missing', 'clothing', 'Нужна синяя куртка.', 'Материал', 'Хлопок', 'unknown'),
    ('clothing_negation', 'clothing', 'Куртка из полиэстера, хлопка в составе нет.', 'Материал', 'Хлопок', 'contradicted'),
    ('clothing_package', 'clothing', 'Куртка из полиэстера без хлопка. Упаковочный мешок изготовлен из хлопка.', 'Материал', 'Хлопок', 'contradicted'),
    ('clothing_correction', 'clothing', 'Куртка из хлопка. Исправление состава: предыдущее указание отменено, материал — полиэстер, хлопка нет.', 'Материал', 'Хлопок', 'contradicted'),
]
fact_followup_jobs = []
for case_id, domain, text, name, value, expected in followup_cases:
    fact = {'name': name, 'value': value}
    fact_followup_jobs.append({
        'id': case_id, 'experiment': 'fact_followup', 'domain': domain,
        'facts': [fact], 'expected': {'fact': expected},
        'baseline': {'fact': ce.lexical_fact_status(text, fact)},
        'body': ce.request_body(text, {'fact': ce.fact_question(fact)}),
    })
followup_jobs = numeric_ab_jobs + fact_followup_jobs

# Validate labels, offsets and that the paired request changes only candidate representation.
assert len({job['id'] for job in followup_jobs}) == len(followup_jobs)
for raw, normalized in zip(numeric_ab_jobs[::2], numeric_ab_jobs[1::2]):
    normalized_body = deepcopy(normalized['body'])
    normalized_body['questions']['voltage']['criteria'] = raw['body']['questions']['voltage']['criteria']
    assert normalized_body == raw['body']
    assert raw['expected'] == normalized['expected']
for job in followup_jobs:
    for question_id, label in job['expected'].items():
        assert label in job['body']['questions'][question_id]['criteria']
    for span in job.get('spans', []):
        assert job['body']['state'][span['start']:span['end']] == span['raw']

print('Prepared:', len(followup_jobs), 'requests;', sum(len(j['body']['questions']) for j in followup_jobs), 'labeled questions')
print('Fact labels:', dict(Counter(j['expected']['fact'] for j in fact_followup_jobs)))
print('Fact domains:', dict(Counter(j['domain'] for j in fact_followup_jobs)))
conversion = next(j for j in numeric_ab_jobs if j['id'] == 'voltage_conversion_normalized')
print('Normalized conversion candidate:', conversion['body']['questions']['voltage']['criteria']['span_0'])

Prepared: 29 requests; 29 labeled questions
Fact labels: {'unknown': 5, 'supported': 4, 'contradicted': 6, 'conflicting': 2}
Fact domains: {'tools': 7, 'furniture': 5, 'clothing': 5}
Normalized conversion candidate: Этот фрагмент задает напряжение аккумулятора самого товара: 0,018 кВ (позиции 30:38). Нормализованное значение: 18.000 В; ОКЕИ 222.


In [12]:
def supported_metrics(records, threshold=THRESHOLD):
    counts = Counter(evaluated=0, errors=0, true_supported=0, predicted_supported=0, correct_supported=0, unsupported_additions=0, accepted_supported=0, accepted_correct_supported=0, accepted_unsupported_additions=0)
    for record in records:
        if record['experiment'] != 'fact_followup':
            continue
        if 'error' in record:
            counts['errors'] += 1
            continue
        for question_id, expected in record['expected'].items():
            answer = record['response']['answers'][question_id]
            positive = answer['choice'] == 'supported'
            correct = positive and expected == 'supported'
            accepted = positive and answer['probabilities']['supported'] >= threshold
            counts['evaluated'] += 1
            counts['true_supported'] += expected == 'supported'
            counts['predicted_supported'] += positive
            counts['correct_supported'] += correct
            counts['unsupported_additions'] += positive and expected != 'supported'
            counts['accepted_supported'] += accepted
            counts['accepted_correct_supported'] += accepted and expected == 'supported'
            counts['accepted_unsupported_additions'] += accepted and expected != 'supported'
    ratio = lambda numerator, denominator: counts[numerator] / counts[denominator] if counts[denominator] else None
    return dict(counts,
        supported_precision=ratio('correct_supported', 'predicted_supported'),
        supported_recall=ratio('correct_supported', 'true_supported'),
        accepted_precision=ratio('accepted_correct_supported', 'accepted_supported'),
        threshold_uncalibrated=threshold)

RUN_FOLLOWUP = False  # set True along with RUN_LIVE only when ready to submit the 29 requests
followup_records = []
followup_summary = None
followup_destination = None
if RUN_LIVE and RUN_FOLLOWUP:
    followup_destination = ROOT / 'experiments/results' / (datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ') + '_followup')
    followup_records, followup_summary = ce.run_jobs(client, followup_jobs, followup_destination, threshold=THRESHOLD)
    report = {
        'supported_facts': supported_metrics(followup_records),
        'numeric_pairs': [],
        'fact_domains': {},
        'note': 'Manually labeled diagnostics; precision excludes API errors, which are counted separately.',
    }
    by_id = {r['id']: r for r in followup_records}
    for original in ce.numeric_jobs():
        pair = {'case': original['id'], 'expected': original['expected']['voltage']}
        for arm in ('raw', 'normalized'):
            record = by_id.get(original['id'] + '_' + arm)
            pair[arm] = record['response']['answers']['voltage'] if record and 'response' in record else {'error': record.get('error', 'not executed') if record else 'not executed'}
        report['numeric_pairs'].append(pair)
    for domain in sorted({j['domain'] for j in fact_followup_jobs}):
        records = [r for r in followup_records if r.get('domain') == domain]
        report['fact_domains'][domain] = {
            'status': ce.summarize(records)['experiments'].get('fact_followup', {}),
            'supported_facts': supported_metrics(records),
        }
    (followup_destination / 'followup_report.json').write_text(json.dumps(report, ensure_ascii=False, indent=2))
    print('Saved:', followup_destination)
    print(json.dumps(followup_summary['experiments'], ensure_ascii=False, indent=2))
    print(json.dumps(report, ensure_ascii=False, indent=2))
else:
    print('Follow-up ready; no inference requests made. Enable RUN_LIVE and RUN_FOLLOWUP when the tunnel is available.')

Follow-up ready; no inference requests made. Enable RUN_LIVE and RUN_FOLLOWUP when the tunnel is available.


Follow-up status: request preparation and label/span validation can run offline. No new model result is claimed until `followup_summary` exists. To execute only the continuation, run setup/imports, catalog loading, and the three code cells in sections 7–8; enable `RUN_LIVE` in setup and `RUN_FOLLOWUP` in the final code cell. Results include `followup_report.json` alongside the usual manifest, health, requests and summary. Compare the paired numeric answers and unsupported additions before changing production extraction.

Execution checkpoint (2026-10-10): all existing offline code cells and the follow-up preparation cells ran successfully. The 29 labels/options, source-span offsets and paired request bodies were validated; supported-fact metric probes covered false additions, abstention, API errors and empty denominators. The local endpoint refused the connection, and a read-only health check through `inh-compute` ended with the SSH connection closed. No follow-up inference has run. Restore the SSH/API connection before enabling the live flags.